---
## 0 — Installation & Imports

In [ ]:
!pip install scikit-learn matplotlib seaborn scipy xgboost tqdm optuna shap --quiet
!pip install lightgbm --config-settings=cmake.define.USE_GPU=ON --quiet 2>/dev/null || \
    pip install lightgbm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.2 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings, subprocess, os
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, brier_score_loss,
    log_loss, classification_report, RocCurveDisplay,
    PrecisionRecallDisplay, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import lightgbm as lgb

np.random.seed(42)

---
## 1 — Configuration

In [ ]:
DATA_PATH    = "/content/drive/MyDrive/collab/f2/india+portugal+historical+population+f.csv"
OUTPUT_DIR   = "/content/drive/MyDrive/collab/f2/v10/"

TARGET_COL   = "in_fire"
RANDOM_STATE = 42

# ── Grille spatiale (Portugal entier) ────────────────────────
N_BLOCKS_X, N_BLOCKS_Y = 4, 4   # 25 blocs

# ── Nested Cross-Validation spatiale ─────────────────────────
K_TEST_BLOCKS    = 3     # blocs TEST par iteration
N_CV_ITERATIONS  = 5     # iterations (chacune a son propre Optuna)

# ── Optuna (1 par iteration) ─────────────────────────────────
N_OPTUNA_TRIALS  = 10    # trials par iteration (peu car x8 iterations)
N_OPTUNA_SPLITS  = 2     # splits par trial
K_VAL_BLOCKS     = 3     # blocs val par split

# ── Divers ───────────────────────────────────────────────────
MIN_FIRES_PER_BLOCK = 0
MAX_FIRE_RATE_TRAIN = 1
RADIUS_KM = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Nested CV: {K_TEST_BLOCKS} blocs test x {N_CV_ITERATIONS} iterations")
print(f"Optuna: {N_OPTUNA_TRIALS} trials x {N_OPTUNA_SPLITS} splits PAR ITERATION")
print(f"Total Optuna fits: ~{N_CV_ITERATIONS * N_OPTUNA_TRIALS * N_OPTUNA_SPLITS}")
print(f"Grille: {N_BLOCKS_X}x{N_BLOCKS_Y} = {N_BLOCKS_X*N_BLOCKS_Y} blocs")

Nested CV: 3 blocs test x 5 iterations
Optuna: 10 trials x 2 splits PAR ITERATION
Total Optuna fits: ~100
Grille: 4x4 = 16 blocs


---
## 2 — Data Loading & Features

In [ ]:

!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
data = pd.read_csv(DATA_PATH)
print(f"Shape: {data.shape} | Fire rate: {data[TARGET_COL].mean():.2%}")
hist_cols = [c for c in data.columns if 'was_in_fire' in c]
print(f"Colonnes historiques: {hist_cols}")

Shape: (89198, 56) | Fire rate: 9.94%
Colonnes historiques: ['was_in_fire_2021', 'was_in_fire_2021_distance', 'was_in_fire_2022', 'was_in_fire_2022_distance', 'was_in_fire_2023', 'was_in_fire_2023_distance']


In [ ]:
# Encode categoricals (user v5)
for col in ['TEXTURE_SOTER']:
    if col in data.columns and data[col].dtype == object:
        data[col] = LabelEncoder().fit_transform(data[col].astype(str))

In [ ]:

# === v9 NEW: Historical fire distance features ===
HIST_FIRE_COLS = ['was_in_fire_2021', 'was_in_fire_2022', 'was_in_fire_2023']
HIST_DIST_COLS = ['was_in_fire_2021_distance', 'was_in_fire_2022_distance', 'was_in_fire_2023_distance']
hist_present = [c for c in HIST_FIRE_COLS if c in data.columns]
dist_present = [c for c in HIST_DIST_COLS if c in data.columns]
print(f"Feux historiques: {hist_present}")
print(f"Distances historiques: {dist_present}")



# Features de distance historique
if len(dist_present) > 0:
    data['min_hist_fire_dist'] = data[dist_present].min(axis=1)
    data['mean_hist_fire_dist'] = data[dist_present].mean(axis=1)




Feux historiques: ['was_in_fire_2021', 'was_in_fire_2022', 'was_in_fire_2023']
Distances historiques: ['was_in_fire_2021_distance', 'was_in_fire_2022_distance', 'was_in_fire_2023_distance']


### 2.2 — Voisinage spatial

In [ ]:
from scipy.spatial import cKDTree
import time



# ══════════════════════════════════════════════════════════════
# FIRE DENSITY: utilise les feux HISTORIQUES (2021-2023)
# Pas de leakage car le target (in_fire) est une annee differente
# ══════════════════════════════════════════════════════════════
HIST_FIRE_COLS = ['was_in_fire_2021', 'was_in_fire_2022', 'was_in_fire_2023']
hist_cols_present = [c for c in HIST_FIRE_COLS if c in data.columns]
print(f"Colonnes historiques trouvees: {hist_cols_present}")


start = time.time()

# Coordonnees en km
mean_lat = np.deg2rad(data['latitude'].mean())
x_km = data['longitude'].values * 111.0 * np.cos(mean_lat)
y_km = data['latitude'].values * 111.0
coords = np.column_stack([x_km, y_km])

tree = cKDTree(coords)
neighbors_list = tree.query_ball_tree(tree, r=RADIUS_KM)



# Feux historiques combines (moyenne 2021-2023)
if len(hist_cols_present) > 0:
    hist_fire = data[hist_cols_present].mean(axis=1).values  # proba historique
else:
    hist_fire = None

n = len(data)
neigh_n_points = np.zeros(n)
neigh_hist_fire_density = np.zeros(n)  # densite feux HISTORIQUES
neigh_forest_pct = np.zeros(n)

for i in range(n):
    nb_idx = neighbors_list[i]
    neigh_n_points[i] = len(nb_idx)
    if len(nb_idx) < 2: continue



    if hist_fire is not None:
        neigh_hist_fire_density[i] = np.mean(hist_fire[nb_idx])


data['neigh_n_points'] = neigh_n_points
data['neigh_forest_pct'] = neigh_forest_pct
data['neigh_hist_fire_density'] = neigh_hist_fire_density

elapsed = time.time() - start
print(f"\nDone en {elapsed:.0f}s — {n:,} points, "
      f"~{np.mean([len(x) for x in neighbors_list]):.0f} voisins/point")
if hist_fire is not None:
    print(f"neigh_hist_fire_density: mean={data['neigh_hist_fire_density'].mean():.3f}, "
          f"max={data['neigh_hist_fire_density'].max():.3f}")

Colonnes historiques trouvees: ['was_in_fire_2021', 'was_in_fire_2022', 'was_in_fire_2023']

Done en 2s — 89,198 points, ~75 voisins/point
neigh_hist_fire_density: mean=0.095, max=0.944


### 2.3 — Configuration des features (modifier ici)

In [ ]:
# ══════════════════════════════════════════════════════════════
# FEATURE CONFIGURATION — modifier ici (True/False ou 1/0)
# ══════════════════════════════════════════════════════════════

FEATURE_GROUPS = {
    # === Sol ===
    'sol_base':          1,   # COARSE, SAND, SILT, CLAY, BULK, ORG_CARBON, PH_WATER
    'sol_cec':           1,   # CEC_SOIL, BSAT, ALUM_SAT, ESP
    'sol_autres':        1,   # GYPSUM, ELEC_COND, CN_RATIO
    'Foret': 1,
    # === Climat ===
    'climat_base':       1,   # prec_ete, tmax_ete, tmax_hiver, tmin_ete
    'climat_diff':       1,   # dif_perc_*, dif_max_*, dif_min_*

    # === Topographie ===
    'topo':              1,   # Elevation
    'population':         1,
    # === Remote sensing ===
    'remote':            1,   # landcover

    # === Historique feux binaire ===
    'hist_fire_binary':  0,   # was_in_fire_2021/2022/2023,

    # === Historique feux distances ===
    'hist_fire_dist':    1,   # was_in_fire_xxxx_distance, min/mean_hist_fire_dist,


    # === Voisinage feux historiques ===
    'voisinage_fire':    1,   # neigh_hist_fire_density
}

# Mapping groupes -> colonnes
GROUP_COLS = {
    'population':    ['population'],
    'Foret' : ['is_Eucalipto','is_Outras_folhosas','is_Pinheiro_bravo','is_Sobreiro'],
    'sol_base':      ['COARSE','SAND','SILT','CLAY','BULK','ORG_CARBON','PH_WATER'],
    'sol_cec':       ['CEC_SOIL','BSAT','ALUM_SAT'],
    'sol_autres':    ['GYPSUM','ELEC_COND','CN_RATIO'],
    'climat_base':   ['prec_ete','tmax_ete','tmax_hiver','tmin_ete'],
    'climat_diff':   [c for c in data.columns if c.startswith('dif_')],
    'topo':          ['Elevation'],
    'remote':        ['landcover'],
    'hist_fire_binary': ['was_in_fire_2021','was_in_fire_2022','was_in_fire_2023'],
    'hist_fire_dist': ['was_in_fire_2021_distance','was_in_fire_2022_distance',
                       'was_in_fire_2023_distance','min_hist_fire_dist',
                       'mean_hist_fire_dist'],
    'voisinage_fire': ['neigh_hist_fire_density'],
}

# Construire FEATURE_COLS
ALWAYS_EXCLUDE = {TARGET_COL, 'latitude', 'longitude', 'band',
                  'block_x', 'block_y', 'block_id', 'TEXTURE_SOTER',
                  'TEXTURE_USDA', 'CEC_CLAY', 'CEC_EFF', 'TCARBON_EQ'}

active = []
for g, on in FEATURE_GROUPS.items():
    if on: active.extend([c for c in GROUP_COLS.get(g,[]) if c in data.columns])
FEATURE_COLS = list(dict.fromkeys(c for c in active if c not in ALWAYS_EXCLUDE))

assert 'latitude' not in FEATURE_COLS and 'longitude' not in FEATURE_COLS
assert TARGET_COL not in FEATURE_COLS
for c in FEATURE_COLS:
    assert 'in_fire' not in c or 'was_in_fire' in c, f"LEAKAGE: {c}"

print(f"Features: {len(FEATURE_COLS)}")
print(f"Actifs: {[g for g,e in FEATURE_GROUPS.items() if e]}")
print(f"Desactives: {[g for g,e in FEATURE_GROUPS.items() if not e]}")
print(f"\n{FEATURE_COLS}")

Features: 38
Actifs: ['sol_base', 'sol_cec', 'sol_autres', 'Foret', 'climat_base', 'climat_diff', 'topo', 'population', 'remote', 'hist_fire_dist', 'voisinage_fire']
Desactives: ['hist_fire_binary']

['COARSE', 'SAND', 'SILT', 'CLAY', 'BULK', 'ORG_CARBON', 'PH_WATER', 'CEC_SOIL', 'BSAT', 'ALUM_SAT', 'GYPSUM', 'ELEC_COND', 'CN_RATIO', 'is_Eucalipto', 'is_Outras_folhosas', 'is_Pinheiro_bravo', 'is_Sobreiro', 'prec_ete', 'tmax_ete', 'tmax_hiver', 'tmin_ete', 'dif_perc_hiver_printemps', 'dif_perc_automne_printemps', 'dif_perc_ete_automne', 'dif_max_automne_printemps', 'dif_max_hiver_printemps', 'dif_min_hiver_printemps', 'dif_min_automne_hiver', 'dif_max_hiver_min_automne', 'Elevation', 'population', 'landcover', 'was_in_fire_2021_distance', 'was_in_fire_2022_distance', 'was_in_fire_2023_distance', 'min_hist_fire_dist', 'mean_hist_fire_dist', 'neigh_hist_fire_density']


In [ ]:
data = data.dropna(subset=[TARGET_COL, 'latitude', 'longitude']).reset_index(drop=True)
print(f"Shape: {data.shape} | Features: {len(FEATURE_COLS)}")

Shape: (89198, 61) | Features: 38


In [ ]:
data[FEATURE_COLS].columns

Index(['COARSE', 'SAND', 'SILT', 'CLAY', 'BULK', 'ORG_CARBON', 'PH_WATER',
       'CEC_SOIL', 'BSAT', 'ALUM_SAT', 'GYPSUM', 'ELEC_COND', 'CN_RATIO',
       'is_Eucalipto', 'is_Outras_folhosas', 'is_Pinheiro_bravo',
       'is_Sobreiro', 'prec_ete', 'tmax_ete', 'tmax_hiver', 'tmin_ete',
       'dif_perc_hiver_printemps', 'dif_perc_automne_printemps',
       'dif_perc_ete_automne', 'dif_max_automne_printemps',
       'dif_max_hiver_printemps', 'dif_min_hiver_printemps',
       'dif_min_automne_hiver', 'dif_max_hiver_min_automne', 'Elevation',
       'population', 'landcover', 'was_in_fire_2021_distance',
       'was_in_fire_2022_distance', 'was_in_fire_2023_distance',
       'min_hist_fire_dist', 'mean_hist_fire_dist', 'neigh_hist_fire_density'],
      dtype='object')

---
## 3 — GPU & Functions

In [ ]:
def detect_gpu():
    try:
        r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
            '--format=csv,noheader'], capture_output=True, text=True)
        if r.returncode == 0: print(f"GPU: {r.stdout.strip()}"); return True
    except FileNotFoundError: pass
    print("CPU mode"); return False

USE_GPU = detect_gpu()
gpu_params = {'device':'gpu','gpu_use_dp':False,'max_bin':255} if USE_GPU else {}

GPU: Tesla T4, 15360 MiB


In [ ]:
def create_optuna_objective(X_all, y_all, block_ids, all_train_blocks,
                            k_val, n_splits, scale_pos, gpu_params, seed):
    """
    Optuna objective avec MULTIPLE splits aleatoires de blocs.
    Chaque trial est evalue sur n_splits configurations spatiales differentes.
    → Les best_params sont ceux qui marchent bien EN MOYENNE partout.
    """
    def objective(trial):
        params = {
            'objective':'binary', 'verbose':-1, 'random_state':seed,
            'n_jobs':1, 'n_estimators':600,
            'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            'num_leaves':        trial.suggest_int('num_leaves', 8, 40),
            'max_depth':         trial.suggest_int('max_depth', 3, 12),
            'min_child_samples': trial.suggest_int('min_child_samples', 50, 500, log=True),
            'reg_alpha':         trial.suggest_float('reg_alpha', 0.1, 20.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
            'feature_fraction':  trial.suggest_float('feature_fraction', 0.3, 0.7),
            'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.5, 0.85),
            'bagging_freq': 1,
            'scale_pos_weight':  trial.suggest_float('scale_pos_weight',
                                                      scale_pos*0.3, scale_pos*1.5),
            **gpu_params,
        }

        scores = []
        for s in range(n_splits):
            # Tirer k_val blocs comme validation
            rng = np.random.RandomState(seed + trial.number * 1000 + s)
            val_blocks = sorted(rng.choice(all_train_blocks,
                                           size=min(k_val, len(all_train_blocks)-1),
                                           replace=False).tolist())
            train_blocks = [b for b in all_train_blocks if b not in val_blocks]

            train_idx = np.isin(block_ids, train_blocks)
            val_idx = np.isin(block_ids, val_blocks)

            if val_idx.sum() == 0 or train_idx.sum() == 0: continue
            if y_all[val_idx].sum() == 0: continue  # pas de feux dans la val

            m = lgb.LGBMClassifier(**params)
            m.fit(X_all[train_idx], y_all[train_idx],
                  eval_set=[(X_all[val_idx], y_all[val_idx])],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(0)])
            scores.append(average_precision_score(
                y_all[val_idx], m.predict_proba(X_all[val_idx])[:,1]))

        return np.mean(scores) if scores else 0.0

    return objective

In [ ]:
def balance_train_blocks(data_region, block_col, target_col, max_fire_rate=0.35, seed=42):
    """
    Sous-echantillonne les positifs dans les blocs ou fire_rate > max_fire_rate.
    Garde tous les negatifs. Retourne un index.
    """
    rng = np.random.RandomState(seed)
    keep_idx = []
    for bid, grp in data_region.groupby(block_col):
        fires = grp[grp[target_col] == 1]
        no_fires = grp[grp[target_col] == 0]
        fr = len(fires) / len(grp) if len(grp) > 0 else 0
        if fr > max_fire_rate and len(fires) > 0:
            # Downsample fires to reach max_fire_rate
            target_n_fires = int(len(no_fires) * max_fire_rate / (1 - max_fire_rate))
            target_n_fires = max(target_n_fires, 10)
            sampled = fires.sample(n=min(target_n_fires, len(fires)), random_state=rng)
            keep_idx.extend(no_fires.index.tolist())
            keep_idx.extend(sampled.index.tolist())
        else:
            keep_idx.extend(grp.index.tolist())
    return sorted(keep_idx)

---
## 4 — Spatial CV Functions

In [ ]:
def create_spatial_blocks(data_region, n_bx, n_by):
    """Cree la grille spatiale et retourne data + stats."""
    data_region = data_region.copy()
    data_region['block_x'] = pd.qcut(data_region['longitude'], q=n_bx, labels=False)
    data_region['block_y'] = pd.qcut(data_region['latitude'], q=n_by, labels=False)
    data_region['block_id'] = data_region['block_y'] * n_bx + data_region['block_x']

    bstats = data_region.groupby('block_id').agg(
        n_pts=(TARGET_COL,'count'), n_fires=(TARGET_COL,'sum'),
        fire_pct=(TARGET_COL,'mean')).reset_index()
    bstats['fire_pct'] = (bstats['fire_pct']*100).round(1)

    all_blocks = bstats[bstats['n_fires'] >= MIN_FIRES_PER_BLOCK]['block_id'].tolist()
    print(f"Grille {n_bx}x{n_by}: {len(bstats)} blocs, {len(all_blocks)} valides")
    return data_region, all_blocks, bstats

In [ ]:
def select_test_blocks(all_blocks, k, seed):
    """Tire K blocs au hasard parmi les blocs valides."""
    rng = np.random.RandomState(seed)
    return sorted(rng.choice(all_blocks, size=min(k, len(all_blocks)), replace=False).tolist())

In [ ]:
def run_optuna_global(data_region, feature_cols, all_blocks, k_test, k_val,
                      n_splits, n_trials, gpu_params, random_state):
    """
    Optuna global avec multi-split:
    - Retire K blocs test (fixes, pas utilises dans Optuna)
    - Chaque trial evalue sur N splits aleatoires des blocs TRAIN restants
    - Best params = ceux qui generalisent le mieux en moyenne
    """
    # Retirer K blocs test pour Optuna (pour pas biaiser)
    test_blocks_optuna = select_test_blocks(all_blocks, k_test, seed=random_state)
    train_blocks = [b for b in all_blocks if b not in test_blocks_optuna]

    # Masque sur tout le train (pas de balance ici — Optuna utilise tout)
    train_mask = data_region['block_id'].isin(train_blocks)

    # Balance
    balanced_idx = balance_train_blocks(
        data_region[train_mask], 'block_id', TARGET_COL,
        max_fire_rate=MAX_FIRE_RATE_TRAIN, seed=random_state)
    balanced_mask = data_region.index.isin(balanced_idx)

    # Prepare
    imputer = SimpleImputer(strategy='median')
    X_train = imputer.fit_transform(data_region.loc[balanced_mask, feature_cols].values)
    y_train = data_region.loc[balanced_mask, TARGET_COL].values
    block_ids_train = data_region.loc[balanced_mask, 'block_id'].values
    train_blocks_actual = sorted(np.unique(block_ids_train).tolist())
    scale_pos = (y_train==0).sum() / max(y_train.sum(), 1)

    print(f"Optuna multi-split: {n_trials} trials x {n_splits} splits/trial")
    print(f"  Blocs exclus (test Optuna): {test_blocks_optuna}")
    print(f"  Blocs train Optuna: {train_blocks_actual} ({len(train_blocks_actual)} blocs)")
    print(f"  {k_val} blocs val par split")
    print(f"  {balanced_mask.sum():,} pts train | scale_pos_weight: {scale_pos:.2f}")
    print(f"  Total fits estimes: ~{n_trials * n_splits}")

    obj = create_optuna_objective(
        X_train, y_train, block_ids_train, train_blocks_actual,
        k_val, n_splits, scale_pos, gpu_params, random_state)

    study = optuna.create_study(direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=random_state))
    study.optimize(obj, n_trials=n_trials, show_progress_bar=True)

    print(f"\nBest mean PR-AUC (across {n_splits} splits): {study.best_value:.4f}")
    print(f"Best params: {study.best_params}")
    return study.best_params, scale_pos

In [ ]:
def run_cv_iteration(data_region, feature_cols, all_blocks, test_block_ids,
                     k_val, n_optuna_splits, n_trials,
                     gpu_params, random_state, iteration):
    """
    1 iteration de nested CV:
    1. Split test/train par blocs
    2. Optuna sur le TRAIN de cette iteration
    3. Train final avec best params
    4. Eval sur test
    """
    test_mask = data_region['block_id'].isin(test_block_ids)
    train_mask = ~test_mask

    # Balance train
    balanced_idx = balance_train_blocks(
        data_region[train_mask], 'block_id', TARGET_COL,
        max_fire_rate=MAX_FIRE_RATE_TRAIN, seed=random_state + iteration)
    balanced_mask = data_region.index.isin(balanced_idx)

    # Prepare
    imputer = SimpleImputer(strategy='median')
    X_all = imputer.fit_transform(data_region[feature_cols].values)
    y_all = data_region[TARGET_COL].values
    X_train, y_train = X_all[balanced_mask], y_all[balanced_mask]
    X_test, y_test = X_all[test_mask], y_all[test_mask]
    block_ids_train = data_region.loc[balanced_mask, 'block_id'].values
    train_blocks = sorted(np.unique(block_ids_train).tolist())
    scale_pos = (y_train==0).sum() / max(y_train.sum(), 1)

    # ── Optuna pour CETTE iteration ──────────────────────────
    obj = create_optuna_objective(
        X_train, y_train, block_ids_train, train_blocks,
        k_val, n_optuna_splits, scale_pos, gpu_params,
        random_state + iteration * 1000)

    study = optuna.create_study(direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=random_state + iteration))
    study.optimize(obj, n_trials=n_trials, show_progress_bar=False)

    best_params = study.best_params
    inner_ap = study.best_value

    # ── Train final avec best params ─────────────────────────
    fp = {'objective':'binary', 'verbose':-1,
          'random_state': random_state + iteration,
          'n_jobs':-1, 'n_estimators':600, 'bagging_freq':1,
          **gpu_params, **best_params}

    nv = max(int(len(X_train)*0.1), 100)
    vi = np.random.choice(len(X_train), nv, replace=False)
    ti = np.setdiff1d(np.arange(len(X_train)), vi)

    model = lgb.LGBMClassifier(**fp)
    model.fit(X_train[ti], y_train[ti],
              eval_set=[(X_train[vi], y_train[vi])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])

    n_trees = model.best_iteration_ if model.best_iteration_ else 400

    # ── Evaluate ─────────────────────────────────────────────
    y_pt = model.predict_proba(X_test)[:,1]
    y_pd = (y_pt >= 0.5).astype(int)

    auc_test = roc_auc_score(y_test, y_pt)
    ap_test = average_precision_score(y_test, y_pt)
    brier = brier_score_loss(y_test, y_pt)
    ll = log_loss(y_test, y_pt)
    f1 = f1_score(y_test, y_pd, zero_division=0)

    ns = min(10_000, len(X_train))
    si = np.random.choice(len(X_train), ns, replace=False)
    auc_train = roc_auc_score(y_train[si], model.predict_proba(X_train[si])[:,1])

    return {
        'iteration': iteration,
        'test_blocks': test_block_ids,
        'n_test': int(test_mask.sum()),
        'n_train': int(balanced_mask.sum()),
        'fire_pct_test': round(y_test.mean()*100, 1),
        'auc_test': auc_test, 'auc_train': auc_train,
        'gap': auc_train - auc_test,
        'ap_test': ap_test, 'brier': brier, 'logloss': ll, 'f1': f1,
        'n_trees': n_trees, 'inner_ap': inner_ap,
        'best_params': best_params,
        'model': model, 'test_mask': test_mask, 'y_proba_test': y_pt,
    }

In [ ]:
def run_spatial_cv(region_name, data_region, feature_cols, n_bx, n_by,
                   k_test, n_iterations, k_val, n_optuna_splits,
                   n_trials, gpu_params, random_state):
    """
    Nested spatial CV:
    1. Create blocks
    2. N iterations, EACH with its own Optuna
    3. Aggregate results
    """
    print(f"\n{'='*70}")
    print(f"  {region_name} | NESTED CV")
    print(f"  {k_test} blocs test x {n_iterations} iterations")
    print(f"  {n_trials} Optuna trials x {n_optuna_splits} splits PAR iteration")
    print(f"{'='*70}")

    # 1. Create blocks
    data_region, all_blocks, bstats = create_spatial_blocks(data_region, n_bx, n_by)
    for _, r in bstats.iterrows():
        flag = "ok" if r['n_fires'] >= MIN_FIRES_PER_BLOCK else "!!"
        print(f"  B{int(r['block_id']):>2}: {int(r['n_pts']):>5,} pts, "
              f"{int(r['n_fires']):>4,} feux ({r['fire_pct']:>5.1f}%) {flag}")

    if len(all_blocks) < k_test + 2:
        print(f"Pas assez de blocs valides")
        return None, data_region

    # 2. Nested CV iterations (each with its own Optuna)
    print(f"\n--- Nested CV ({n_iterations} iterations) ---")
    cv_results = []
    for i in range(n_iterations):
        seed_i = random_state + i * 100
        test_blocks_i = select_test_blocks(all_blocks, k_test, seed=seed_i)

        print(f"\n  Iter {i+1}/{n_iterations}: test={test_blocks_i}")
        print(f"    Optuna {n_trials} trials...", end=" ")

        res_i = run_cv_iteration(
            data_region, feature_cols, all_blocks, test_blocks_i,
            k_val, n_optuna_splits, n_trials,
            gpu_params, random_state, iteration=i)

        cv_results.append(res_i)
        print(f"inner={res_i['inner_ap']:.4f} → "
              f"AUC={res_i['auc_test']:.4f} | PR-AUC={res_i['ap_test']:.4f} "
              f"| Gap={res_i['gap']:.4f} | Trees={res_i['n_trees']}")

    # 3. Aggregate
    cv_df = pd.DataFrame([{k:v for k,v in r.items()
        if k not in ['model','test_mask','y_proba_test','test_blocks','best_params']}
        for r in cv_results])

    print(f"\n{'─'*70}")
    print(f"  {region_name} — SUMMARY ({n_iterations} iterations)")
    print(f"{'─'*70}")
    for m in ['auc_test', 'ap_test', 'brier', 'logloss', 'f1', 'gap', 'inner_ap']:
        v = cv_df[m]
        print(f"  {m:<12}: {v.mean():.4f} +/- {v.std():.4f}")
    print(f"  trees (avg) : {cv_df['n_trees'].mean():.0f}")

    return {
        'cv_results': cv_results,
        'cv_df': cv_df,
        'all_blocks': all_blocks,
        'bstats': bstats,
    }, data_region

---
## 5 — Run (Portugal entier)

In [ ]:
results, data = run_spatial_cv(
    "PORTUGAL", data, FEATURE_COLS,
    N_BLOCKS_X, N_BLOCKS_Y,
    K_TEST_BLOCKS, N_CV_ITERATIONS,
    K_VAL_BLOCKS, N_OPTUNA_SPLITS,
    N_OPTUNA_TRIALS, gpu_params, RANDOM_STATE)


  PORTUGAL | NESTED CV
  3 blocs test x 5 iterations
  10 Optuna trials x 2 splits PAR iteration
Grille 4x4: 16 blocs, 16 valides
  B 0: 5,596 pts,  271 feux (  4.8%) ok
  B 1: 6,876 pts,  262 feux (  3.8%) ok
  B 2: 7,097 pts,  393 feux (  5.5%) ok
  B 3: 2,731 pts,  128 feux (  4.7%) ok
  B 4: 8,972 pts,  483 feux (  5.4%) ok
  B 5: 5,043 pts,   65 feux (  1.3%) ok
  B 6: 5,156 pts,  170 feux (  3.3%) ok
  B 7: 3,128 pts,  102 feux (  3.3%) ok
  B 8: 4,464 pts,  399 feux (  8.9%) ok
  B 9: 5,416 pts,  666 feux ( 12.3%) ok
  B10: 5,442 pts,  836 feux ( 15.4%) ok
  B11: 6,977 pts,  306 feux (  4.4%) ok
  B12: 3,268 pts,  647 feux ( 19.8%) ok
  B13: 4,964 pts, 2,307 feux ( 46.5%) ok
  B14: 4,604 pts, 1,271 feux ( 27.6%) ok
  B15: 9,464 pts,  556 feux (  5.9%) ok

--- Nested CV (5 iterations) ---

  Iter 1/5: test=[0, 1, 5]
    Optuna 10 trials... inner=0.4893 → AUC=0.8067 | PR-AUC=0.2317 | Gap=0.0904 | Trees=43

  Iter 2/5: test=[6, 9, 13]
    Optuna 10 trials... inner=0.4253 → AUC=0.8